# 🎓 졸업요건 RAG 챗봇 (다면적졸업인증제)

> **LLM 과목 과제**로 진행한 프로젝트입니다. `index.html`(웹 챗봇)과 같은 지식베이스·규칙을 파이썬으로 실습하는 노트북입니다.

**구성**
1. 지식베이스 로드 — `knowledge/*.md` (PDF를 학과·학번별로 정리한 chunk)
2. 검색기(Retriever) — BM25 + 학과·학번 메타데이터 필터 (외부 API 없음)
3. 질의응답 — 근거 chunk를 그대로 보여주는 추출형 RAG (환각 없음)
4. 졸업 판정 — 수강 과목·활동을 입력하면 구간별 점수 계산 (6개 학과)
5. 자동 검증

**기준 문서**: `docs/2026년도 학과기준 디자인 시안(1차 시안-SW융합대학).pdf`

> 먼저 터미널에서 `python3 build.py`를 한 번 실행해 `data/checker_rules.json`을 만들어 두세요.

In [ ]:
import json, math, re, sys
from collections import Counter
from pathlib import Path

# 이 노트북은 프로젝트 폴더(졸업요건_RAG_챗봇) 안에서 실행한다고 가정한다.
ROOT = Path.cwd()
if not (ROOT / "knowledge").exists():
    ROOT = next(p for p in [ROOT.parent, *ROOT.glob("*/")] if (p / "knowledge").exists())
sys.path.insert(0, str(ROOT))
print("프로젝트 폴더:", ROOT)

## 1. 지식베이스 로드

`build.py`의 `parse_knowledge`를 재사용합니다. 각 chunk는 제목·본문과 함께 `dept`(학과), `years`(적용 학번 범위), `page`(PDF 쪽수) 메타데이터를 가집니다.

In [ ]:
from build import parse_knowledge

chunks = []
for md_file in sorted((ROOT / "knowledge").glob("*.md")):
    chunks += parse_knowledge(md_file)

DEPTS = {"cse": "컴퓨터소프트웨어공학과", "infosec": "정보보호학과", "medit": "의료IT공학과",
         "aibd": "AI·빅데이터학과", "iot": "사물인터넷학과", "meta": "메타버스&게임학과"}

print(f"chunk {len(chunks)}개")
for c in chunks[:3]:
    print(f"- [{c['dept']}] {c['years']} p.{c['page']}  {c['title']}")
print()
print(chunks[4]["text"])

## 2. 검색기 (BM25 + 메타데이터 필터)

1. 질문에서 **학과·학번**을 읽어 맞는 chunk에 가중치를 주고, 다른 학과/학번 chunk는 낮춥니다.
2. 한글은 **2글자 조각 + 단어**로 토큰화합니다. 형태소 분석기 없이도 조사 변화에 강합니다.
3. BM25로 순위를 매기고, 질문의 핵심 단어가 근거 문서에 충분히 있는지(coverage) 확인해 **범위 밖 질문은 답하지 않습니다.**

In [ ]:
DEPT_PATTERNS = {
    "cse": r"컴퓨터\s*소프트웨어(공학)?|컴소공|컴소|소프트웨어공학",
    "infosec": r"정보\s*보호|정보보안",
    "medit": r"의료\s*it|의료\s*아이티",
    "aibd": r"ai\s*[·.\-ㆍ]?\s*빅\s*데이터|빅데이터\s*학과|에이아이",
    "iot": r"사물\s*인터넷|iot",
    "meta": r"메타\s*버스|게임\s*학과",
}
FILLER = {"가요","돼요","되나","되나요","있나","있나요","알려","려줘","줘요","주세","세요","뭐","뭔가","어떻","어떻게","하나","하나요","할까","얼마","필요","요건","해야","야해","게해","뭐야"}
KO_SUFFIX = re.compile(r"(이란|이야|인가요|인가|나요|가요|어요|해요|해줘|알려줘|알려주세요|뭐야|뭐예요|입니까|이에요|예요|에서|으로|까지|부터|이|가|은|는|을|를|의|에|로|와|과|도|만)$")

def detect_dept(q):
    return next((d for d, p in DEPT_PATTERNS.items() if re.search(p, q, re.I)), None)

def detect_year(q):
    m = re.search(r"(20\d{2})\s*(학번|년\s*입학|입학)", q) or re.search(r"(?<!\d)(\d{2})\s*학번", q)
    if m:
        y = int(m.group(1)); return y if y > 100 else 2000 + y
    m = re.search(r"(20\d{2})", q)
    return int(m.group(1)) if m else None

def tokenize(text):
    out = []
    for w in re.findall(r"[가-힣]+|[a-z0-9]+", text.lower()):
        if re.fullmatch(r"[가-힣]+", w):
            if len(w) >= 3:
                w = KO_SUFFIX.sub("", w) or w
            if len(w) == 1:
                out.append(w); continue
            out += [w[i:i+2] for i in range(len(w) - 1)]
            if len(w) >= 3: out.append(w)
        else:
            out.append(w)
    return out

class Retriever:
    def __init__(self, chunks):
        self.chunks = chunks
        self.docs = [tokenize(f"{c['title']} {c['title']} {c['text']}") for c in chunks]
        self.avg = sum(map(len, self.docs)) / len(self.docs)
        self.tf = [Counter(d) for d in self.docs]
        self.df = Counter(t for tf in self.tf for t in tf)

    def bm25(self, q_tokens, i, k1=1.4, b=0.75):
        N, s, ln = len(self.chunks), 0.0, len(self.docs[i])
        for t in set(q_tokens):
            f = self.tf[i].get(t)
            if f:
                n = self.df[t]
                s += math.log(1 + (N - n + 0.5) / (n + 0.5)) * f * (k1 + 1) / (f + k1 * (1 - b + b * ln / self.avg))
        return s

    def search(self, question, profile=None, top_k=3):
        profile = profile or {}
        asked_dept, asked_year = detect_dept(question), detect_year(question)
        dept = asked_dept or profile.get("dept")
        other = asked_dept and profile.get("dept") and asked_dept != profile["dept"]
        year = asked_year or (None if other else profile.get("year"))

        cleaned = re.sub(r"(20\d{2}|\d{2})\s*(학번|년\s*입학)", " ", question)
        cleaned = re.sub("|".join(DEPT_PATTERNS.values()), " ", cleaned, flags=re.I).replace("학과", " ")
        cleaned = re.sub(r"총\s*점수?|졸업\s*점수|필요(한|하)?\s*점수|기준\s*점수|몇\s*점|몇점", " 합계 ", cleaned)
        q_tokens = [t for t in tokenize(cleaned) if t not in FILLER]
        wants_guide = bool(re.search(r"어디|문의|연락|위치|방문|센터|신청|확인\s*(방법|하는)", question))

        scored = []
        for i, c in enumerate(self.chunks):
            raw = self.bm25(q_tokens, i)
            if raw <= 0: continue
            mult = 1.0
            if wants_guide and c["dept"] == "common" and not c["years"]: mult *= 2
            if c["dept"] not in (None, "common") and dept:
                mult *= 1.5 if c["dept"] == dept else (0.05 if asked_dept else 0.4)
            if c["years"] and year:
                a, b = c["years"]
                mult *= 1.6 if a <= year <= b else (0.1 if asked_year else 0.35)
            scored.append((raw * mult, raw, i))
        scored.sort(reverse=True)
        if not scored:
            return {"results": [], "confident": False}
        top = scored[0]
        results = [self.chunks[i] for s, _, i in scored if s >= top[0] * 0.5][:top_k]
        uniq = set(q_tokens)
        coverage = len(uniq & set(self.tf[top[2]])) / len(uniq) if uniq else 0
        dept_hit = bool(asked_dept) and self.chunks[top[2]]["dept"] == asked_dept
        return {"results": results, "dept": dept, "year": year, "coverage": coverage,
                "confident": coverage >= 0.34 and (top[1] >= 0.3 or dept_hit),
                "need_year": not year and any(c["years"] for c in results)}

retriever = Retriever(chunks)

## 3. 질의응답 (추출형 RAG)

생성형 모델로 문장을 지어내지 않고, **검색된 근거 chunk의 원문과 출처(쪽수)** 를 그대로 보여줍니다.

In [ ]:
def ask(question, profile={"dept": "cse", "year": 2022}):
    r = retriever.search(question, profile)
    print(f"Q. {question}")
    if not r["results"] or not r["confident"]:
        print("A. 문서에서 관련 내용을 찾지 못했어요. 학과와 학번을 함께 적어 보세요.\n")
        return
    top = r["results"][0]
    m = re.search(r"합계\s*(\d+)\s*점", top["text"])
    if m and re.search(r"총\s*점|합계|몇\s*점|필요(한|하)?\s*점수", question) and top["dept"] not in (None, "common"):
        print(f"A. {DEPTS[top['dept']]} 졸업 기준은 합계 {m.group(1)}점입니다.")
    if r["need_year"]:
        print("   (학번을 함께 적으면 그 학번 기준만 보여드려요)")
    for c in r["results"]:
        print(f"\n 📎 {c['title']}  (p.{c['page']})")
        print("   " + c["text"].replace("\n", "\n   "))
    print()

ask("2022학번 컴퓨터소프트웨어공학과 졸업 총점은?")
ask("정보보호학과 2023학번 필요 점수")
ask("비교과 신청은 어디서 해요?")
ask("장학금 신청 기간")   # 범위 밖 → 답하지 않음

## 4. 졸업 판정

`data/checker_rules.json`(웹 챗봇과 **같은 규칙**)을 읽어 점수를 계산합니다. 말로 추론하지 않고 기준표로 직접 계산하므로 결과가 재현됩니다.

- 구간: 대학공통 → 교과 → 비교과 지정 → 비교과 미지정
- 구간별 점수는 그 구간 요구 점수까지만 반영
- 과목명은 띄어쓰기·로마숫자(Ⅱ)·부분 일치를 허용 (가장 긴 별칭 우선: `산학캡스톤디자인` ≠ `캡스톤디자인`)

In [ ]:
RULES = json.loads((ROOT / "data" / "checker_rules.json").read_text(encoding="utf-8"))
ROMAN = str.maketrans("ⅠⅡⅢⅣⅰⅱⅲⅳ", "12341234")

def norm(s):
    return re.sub(r"[^가-힣a-z0-9]", "", str(s).translate(ROMAN).lower())

def common_key(year):
    return "2024-2026" if year >= 2024 else "2023-2023" if year == 2023 else "2019-2022"

def find_group(dept, year):
    return next((g for g in RULES["groups"] if g["dept"] == dept and g["years"][0] <= year <= g["years"][1]), None)

def match_courses(user_courses, pool):
    matched, unmatched = {}, []
    for raw in user_courses:
        n = norm(raw)
        if not n: continue
        best, best_len = None, 0
        for it in pool:
            if it["id"] in matched: continue
            for a in it["alias"]:
                if norm(a) and norm(a) in n and len(norm(a)) > best_len:
                    best, best_len = it, len(norm(a))
        if best: matched[best["id"]] = raw
        else: unmatched.append(raw)
    return matched, unmatched

def act_line(it, act):
    v = act.get(it["id"], 0)
    if it.get("type") == "flag":
        ok = bool(v); return dict(label=it["label"], earned=it["pts"] if ok else 0, max=it["pts"], ok=ok)
    if it["type"] == "number":
        raw = max(0, float(v or 0)); cap = it.get("max")
        val = min(raw, cap) if cap is not None else raw
        return dict(label=it["label"], earned=val, max=cap, ok=(val >= cap) if cap is not None else val > 0)
    cnt = max(0, int(v or 0)); cap = None if it["max"] >= 99 else it["max"]
    val = cnt * it["per"] if cap is None else min(cnt * it["per"], cap)
    return dict(label=f"{it['label']} ({cnt}회)", earned=val, max=cap, ok=(val >= cap) if cap else val > 0)

def evaluate(dept, year, common_done, courses, act):
    g = find_group(dept, year)
    if not g:
        return {"error": f"{DEPTS.get(dept, dept)} {year}학번은 기준표가 없어 판정할 수 없습니다."}
    cats = []
    cats.append(("대학공통 필수", 45, [dict(label=l, earned=p if l in common_done else 0, max=p, ok=l in common_done)
                                  for l, p in RULES["common"][common_key(year)]]))
    pool = list(g["course"]["items"]); sp = g["course"].get("special")
    if sp: pool += sp["main"] + sp["subs"]
    matched, unmatched = match_courses(courses, pool)
    lines = [dict(label=it["label"], earned=it["pts"] if it["id"] in matched else 0, max=it["pts"], ok=it["id"] in matched)
             for it in g["course"]["items"]]
    if sp:  # 전공영어 택3 (대체과목 인정)
        n = min(sp["slots"], sum(i["id"] in matched for i in sp["main"] + sp["subs"]))
        lines.append(dict(label=sp["label"], earned=n * sp["per"], max=sp["slots"] * sp["per"], ok=n == sp["slots"]))
    cats.append(("교과", g["course"]["req"], lines))
    cats.append(("비교과 지정", g["designated"]["req"], [act_line(i, act) for i in g["designated"]["items"]]))
    cats.append(("비교과 미지정", g["free"]["req"], [act_line(i, act) for i in g["free"]["items"]]))
    result, total, all_ok = [], 0, True
    for name, req, ls in cats:
        earned = min(sum(l["earned"] for l in ls), req)
        total += earned; all_ok &= earned >= req
        result.append(dict(name=name, required=req, earned=earned, lines=ls))
    return dict(group=g, cats=result, total=total, target=g["total"], ok=total >= g["total"] and all_ok,
                remaining=max(0, g["total"] - total), unmatched=unmatched, note=g.get("note"))

def show_inputs(dept, year):
    """활동 입력 키(id)와 의미를 보여준다."""
    g = find_group(dept, year)
    for part in ("designated", "free"):
        for it in g[part]["items"]:
            print(f"  {it['id']:<12} {it['label']}  [{it.get('type')}]")

def print_result(res):
    if "error" in res: print(res["error"]); return
    print(("🎉 충족" if res["ok"] else "⚠️ 미충족") + f"  {res['total']} / {res['target']}점  (부족 {res['remaining']}점)")
    for c in res["cats"]:
        print(f"\n[{c['name']}] {c['earned']} / {c['required']}점")
        for l in c["lines"]:
            mx = "" if l["max"] is None else f"/{l['max']:g}"
            print(f"   {'✔' if l['ok'] else '✘'} {l['label']}  {l['earned']:g}{mx}점")
    if res["unmatched"]: print("\n기준표에서 찾지 못한 과목:", res["unmatched"])
    if res["note"]: print("\nℹ️", res["note"])

### 내 정보 입력

- `dept`, `year`: 학과·입학 학번
- `my_courses`: 수강한 과목 리스트
- `my_activities`: 활동 입력값 — 키는 아래 `show_inputs()`로 확인 (횟수는 정수, 인증/발표 등은 `True`, 점수 입력은 숫자)

In [ ]:
dept, year = "cse", 2022
show_inputs(dept, year)

In [ ]:
# 대학공통 필수: 이수한 항목 이름 (모두 이수했다고 가정하려면 아래 한 줄 그대로)
common_done = {label for label, _ in RULES["common"][common_key(year)]}

my_courses = ["캡스톤디자인", "산학캡스톤디자인", "전공영어기초", "전공영어응용"]

my_activities = {
    "adv": 4,          # 전공상담 횟수
    "fest": True,      # 학술제 발표
    "coding": True,    # 코딩역량인증제 Level2 이상
    "aisw": False,     # AI·SW전공역량졸업인증제
    "topcit": 2,       # TOPCIT 응시 횟수
    "free": 0,         # 기타 비교과자율활동 점수
}

print_result(evaluate(dept, year, common_done, my_courses, my_activities))

## 5. 자동 검증

노트북의 검색기·판정이 기대대로 동작하는지 확인합니다. (웹 버전은 `node tests/run_tests.js`)

In [ ]:
# --- 검색 ---
QA = [
    ("2022학번 컴퓨터소프트웨어공학과 졸업 총점은?", "컴퓨터소프트웨어공학과 2022~2024"),
    ("정보보호학과 2023학번 필요 점수", "정보보호학과 2023~2026"),
    ("의료IT공학과 졸업요건 알려줘 2022", "의료IT공학과 2022~2023"),
    ("사물인터넷 25학번 TOPCIT 몇 점", "사물인터넷학과 2025"),
    ("메타버스 TOPCIT 수준별 점수", "TOPCIT 수준별"),
    ("대학공통 24학번 학습윤리", "2024~2026"),
    ("비교과 신청 어디서", "비교과 신청"),
    ("상담 어디로 가요", "문의처"),
    ("학식 메뉴 뭐야", None), ("장학금 신청 기간", None), ("기숙사 입사 신청", None),
]
ok = 0
for q, expected in QA:
    r = retriever.search(q, {"dept": "cse", "year": 2022})
    got = r["results"][0]["title"] if r["confident"] and r["results"] else None
    good = got is None if expected is None else bool(got and expected in got)
    ok += good
    print("OK  " if good else "FAIL", q, "->", got)
print(f"검색 {ok}/{len(QA)}")

# --- 판정: 모든 학과·그룹에서 전부 이수하면 목표 점수, 미입력이면 0점 ---
bad = 0
for g in RULES["groups"]:
    y = g["years"][0]
    courses = [it["alias"][0] for it in g["course"]["items"]]
    if g["course"].get("special"): courses += [it["alias"][0] for it in g["course"]["special"]["main"]]
    act = {it["id"]: (True if it.get("type") == "flag" else 99) for part in ("designated", "free") for it in g[part]["items"]}
    full = evaluate(g["dept"], y, {l for l, _ in RULES["common"][common_key(y)]}, courses, act)
    none = evaluate(g["dept"], y, set(), [], {})
    if not (full["total"] == g["total"] and full["ok"] and none["total"] == 0):
        bad += 1; print("FAIL", g["dept"], g["years"], full["total"], g["total"])
print(f"판정 그룹 {len(RULES['groups'])}개 중 이상 {bad}개")
assert ok == len(QA) and bad == 0
print("✅ 모든 검증 통과")

## 6. 한계와 확장

- **검색**: 단어 일치 기반(BM25)이라 표현이 크게 다른 질문에는 약합니다. `sentence-transformers` 등 **임베딩 검색**으로 바꾸거나 BM25와 섞는(하이브리드) 방식으로 확장할 수 있습니다.
- **생성**: 지금은 근거 원문을 그대로 보여주는 추출형입니다. 검색된 chunk를 프롬프트에 넣어 **LLM이 자연어로 답하게** 하되, "근거에 없으면 모른다고 답하라"는 제약을 주는 것이 다음 단계입니다.
- **판정 가정**: PDF 표만으로 불분명한 부분(교과 구성, 비교과 미지정 반영 범위 등)은 README의 "지원 범위와 한계"를 참고하세요. 최종 확인은 종합정보시스템의 개인별졸업진도관리에서 하세요.